# Modular Component-Wise Training: Visual Alignment & Multimodal Fusion Network

Notebook này triển khai chiến lược **Huấn luyện theo từng thành phần (Modular Component-Wise Training)** cho hệ thống **Multimodal Lecture Summarizer**:

### Tóm tắt Chiến lược Huấn luyện:
1. **Đóng đóng (Frozen) các mô hình nền tảng khổng lồ (Heavy Backbones & LLM):**
   - **LLM Tóm tắt (DeepSeek / GPT-4 / Qwen / Llama):** Đóng vai trò sinh văn bản tóm tắt qua Prompt Engineering / API, không fine-tune để tiết kiệm VRAM & tránh quá tải tính toán.
   - **Speech-to-Text (Whisper / WhisperX):** Dùng trọng số pre-trained để nhận dạng & căn chỉnh từ (Word Alignment).
   - **Text & Visual Embedders (PhoBERT / SBERT, CLIP ViT backbone):** Đóng băng trọng số, chỉ trích xuất các vector đặc trưng tĩnh (`visual_emb: 512d`, `text_emb: 768d`).

2. **Huấn luyện từng thành phần nhẹ & Mô đun Hình ảnh / Dung hợp (Trainable Visual & Fusion Modules):**
   - **Component 1 (Visual Matcher - `KeyframeMatcher`):** Huấn luyện mô đun căn chỉnh/khớp đặc trưng hình ảnh keyframe (CLIP) với đoạn thoại transcript bài giảng.
   - **Component 2 (Slide Text Matcher - `SlideTextMatcher`):** Huấn luyện mô đun căn chỉnh văn bản trên slide (OCR) với thoại transcript.
   - **Component 3 (Multimodal Scene Encoder - `MultimodalSceneEncoder`):** Huấn luyện mạng Cross-modal Attention và lớp dung hợp (Fusion Layer) sử dụng hàm lỗi tương phản **InfoNCE Contrastive Loss**.

---

In [ ]:
# 0. Import Thư viện & Cấu hình Thiết bị (Device Setup)
import sys
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Cấu hình GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] Sử dụng thiết bị: {device}")
if torch.cuda.is_available():
    print(f"[GPU] GPU Name: {torch.cuda.get_device_name(0)}")

# Robust Project Root Path detection
curr_p = Path(".").resolve()
if (curr_p / "experiments").exists():
    project_root = curr_p
elif (curr_p.parent / "experiments").exists():
    project_root = curr_p.parent
else:
    project_root = curr_p.parent.parent.resolve()

models_dir = project_root / "storage" / "models"
outputs_dir = project_root / "experiments" / "outputs"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

print(f"Thư mục dự án: {project_root}")
print(f"Thư mục lưu mô hình: {models_dir}")

## Bước 1: Giả lập Dữ liệu Đặc trưng Tĩnh (Pre-extracted Static Features)

Vì các mô hình nặng (CLIP, PhoBERT, Whisper) đã được **đóng băng (Frozen)**, quá trình huấn luyện chỉ thao tác trên ma trận đặc trưng (Embeddings) được trích xuất trước:
- `visual` (512-dim): Đặc trưng thị giác trích xuất từ CLIP ViT-B/32.
- `ocr` (768-dim): Đặc trưng văn bản slide trích xuất từ PhoBERT/SBERT.
- `transcript` (768-dim): Đặc trưng đoạn thoại trích xuất từ PhoBERT/SBERT.

In [ ]:
class MultimodalFeatureDataset(Dataset):
    def __init__(self, num_samples=1200, clip_dim=512, text_dim=768):
        self.num_samples = num_samples
        np.random.seed(42)
        
        # Vector ngữ nghĩa cơ sở (base semantic vector)
        base_features = np.random.randn(num_samples, text_dim).astype(np.float32)
        
        # Transcript embeddings (gần base)
        self.transcript_embeddings = base_features + 0.1 * np.random.randn(num_samples, text_dim).astype(np.float32)
        # OCR embeddings (slide text)
        self.ocr_embeddings = base_features + 0.25 * np.random.randn(num_samples, text_dim).astype(np.float32)
        
        # Visual embeddings (CLIP 512d) được chiếu từ base_features
        proj_w = np.random.randn(text_dim, clip_dim).astype(np.float32) / np.sqrt(text_dim)
        self.visual_embeddings = np.dot(base_features, proj_w) + 0.1 * np.random.randn(num_samples, clip_dim).astype(np.float32)
        
    def __len__(self):
        return self.num_samples
        
    def __getitem__(self, idx):
        return {
            "visual": torch.tensor(self.visual_embeddings[idx], dtype=torch.float32),
            "ocr": torch.tensor(self.ocr_embeddings[idx], dtype=torch.float32),
            "transcript": torch.tensor(self.transcript_embeddings[idx], dtype=torch.float32)
        }

# Khởi tạo Dataset & DataLoader
full_dataset = MultimodalFeatureDataset(num_samples=1200)
train_size = 1000
val_size = 200

train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"[OK] Đã khởi tạo Dataset thành công: {train_size} mẫu Train | {val_size} mẫu Val")

## Bước 2: Huấn luyện Thành phần 1 - Keyframe Visual Matcher (`KeyframeMatcher`)

Mô đun này huấn luyện một bộ phân loại nhị phân (Binary Classifier) để học độ tương quan giữa **Vector hình ảnh Keyframe (CLIP 512d)** và **Vector đoạn thoại Transcript (768d)** nhằm lọc ra các hình ảnh thực sự minh họa cho nội dung bài giảng.

In [ ]:
class KeyframeMatcher(nn.Module):
    """Binary classifier matching Keyframe visual features with Transcript."""
    def __init__(self, clip_dim: int = 512, text_dim: int = 768, hidden_dim: int = 256):
        super(KeyframeMatcher, self).__init__()
        self.proj_visual = nn.Sequential(nn.Linear(clip_dim, hidden_dim), nn.ReLU())
        self.proj_text = nn.Sequential(nn.Linear(text_dim, hidden_dim), nn.ReLU())
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, visual_emb: torch.Tensor, transcript_emb: torch.Tensor) -> torch.Tensor:
        v_proj = self.proj_visual(visual_emb)
        t_proj = self.proj_text(transcript_emb)
        diff = torch.abs(v_proj - t_proj)
        mult = v_proj * t_proj
        features = torch.cat([v_proj, t_proj, diff, mult], dim=-1)
        return self.fc(features).squeeze(-1)

# Training KeyframeMatcher
keyframe_model = KeyframeMatcher().to(device)
optimizer_kf = optim.AdamW(keyframe_model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion_bce = nn.BCELoss()

epochs_kf = 15
losses_train_kf, losses_val_kf = [], []

print("[Train] Huấn luyện Thành phần 1: KeyframeMatcher...")
for epoch in range(1, epochs_kf + 1):
    keyframe_model.train()
    total_loss = 0.0
    for batch in train_loader:
        v = batch["visual"].to(device)
        t = batch["transcript"].to(device)
        t_neg = torch.roll(t, shifts=1, dims=0)
        
        v_pair = torch.cat([v, v], dim=0)
        t_pair = torch.cat([t, t_neg], dim=0)
        labels = torch.cat([torch.ones(v.size(0)), torch.zeros(v.size(0))], dim=0).to(device)
        
        optimizer_kf.zero_grad()
        preds = keyframe_model(v_pair, t_pair)
        loss = criterion_bce(preds, labels)
        loss.backward()
        optimizer_kf.step()
        total_loss += loss.item() * v_pair.size(0)
        
    avg_train = total_loss / (2 * len(train_dataset))
    losses_train_kf.append(avg_train)
    
    # Val
    keyframe_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            v = batch["visual"].to(device)
            t = batch["transcript"].to(device)
            t_neg = torch.roll(t, shifts=1, dims=0)
            v_pair = torch.cat([v, v], dim=0)
            t_pair = torch.cat([t, t_neg], dim=0)
            labels = torch.cat([torch.ones(v.size(0)), torch.zeros(v.size(0))], dim=0).to(device)
            preds = keyframe_model(v_pair, t_pair)
            val_loss += criterion_bce(preds, labels).item() * v_pair.size(0)
    avg_val = val_loss / (2 * len(val_dataset))
    losses_val_kf.append(avg_val)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs_kf:02d}] -> Train BCE: {avg_train:.4f} | Val BCE: {avg_val:.4f}")

# Lưu trọng số KeyframeMatcher
torch.save(keyframe_model.state_dict(), models_dir / "keyframe_matcher.pth")
print(f"[Saved] Đã lưu thành công: {models_dir / 'keyframe_matcher.pth'}")

## Bước 3: Huấn luyện Thành phần 2 - Slide Text Matcher (`SlideTextMatcher`)

Mô đun này huấn luyện mô hình nhị phân đo độ khớp giữa **Văn bản OCR trên Slide (768d)** và **Lời giảng Transcript (768d)**.

In [ ]:
class SlideTextMatcher(nn.Module):
    """Separate binary classifier matching Slide OCR with Transcript."""
    def __init__(self, text_dim: int = 768, hidden_dim: int = 256):
        super(SlideTextMatcher, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(text_dim * 4, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, ocr_emb: torch.Tensor, transcript_emb: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(ocr_emb - transcript_emb)
        mult = ocr_emb * transcript_emb
        features = torch.cat([ocr_emb, transcript_emb, diff, mult], dim=-1)
        return self.fc(features).squeeze(-1)

# Training SlideTextMatcher
slide_model = SlideTextMatcher().to(device)
optimizer_slide = optim.AdamW(slide_model.parameters(), lr=1e-3, weight_decay=1e-2)

epochs_slide = 15
losses_train_slide, losses_val_slide = [], []

print("[Train] Huấn luyện Thành phần 2: SlideTextMatcher...")
for epoch in range(1, epochs_slide + 1):
    slide_model.train()
    total_loss = 0.0
    for batch in train_loader:
        o = batch["ocr"].to(device)
        t = batch["transcript"].to(device)
        t_neg = torch.roll(t, shifts=1, dims=0)
        
        o_pair = torch.cat([o, o], dim=0)
        t_pair = torch.cat([t, t_neg], dim=0)
        labels = torch.cat([torch.ones(o.size(0)), torch.zeros(o.size(0))], dim=0).to(device)
        
        optimizer_slide.zero_grad()
        preds = slide_model(o_pair, t_pair)
        loss = criterion_bce(preds, labels)
        loss.backward()
        optimizer_slide.step()
        total_loss += loss.item() * o_pair.size(0)
        
    avg_train = total_loss / (2 * len(train_dataset))
    losses_train_slide.append(avg_train)
    
    # Val
    slide_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            o = batch["ocr"].to(device)
            t = batch["transcript"].to(device)
            t_neg = torch.roll(t, shifts=1, dims=0)
            o_pair = torch.cat([o, o], dim=0)
            t_pair = torch.cat([t, t_neg], dim=0)
            labels = torch.cat([torch.ones(o.size(0)), torch.zeros(o.size(0))], dim=0).to(device)
            preds = slide_model(o_pair, t_pair)
            val_loss += criterion_bce(preds, labels).item() * o_pair.size(0)
    avg_val = val_loss / (2 * len(val_dataset))
    losses_val_slide.append(avg_val)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs_slide:02d}] -> Train BCE: {avg_train:.4f} | Val BCE: {avg_val:.4f}")

# Lưu trọng số SlideTextMatcher
torch.save(slide_model.state_dict(), models_dir / "slide_matcher.pth")
print(f"[Saved] Đã lưu thành công: {models_dir / 'slide_matcher.pth'}")

## Bước 4: Huấn luyện Thành phần 3 - Multimodal Scene Fusion Encoder (`MultimodalSceneEncoder`)

Mô đun này ứng dụng cơ chế **Cross-modal Attention** (Visual chú ý đến OCR + Transcript) và học tương phản **InfoNCE Loss** để tạo vector biểu diễn dung hợp đa phương thức `joint_embedding` (256d).

In [ ]:
class MultimodalSceneEncoder(nn.Module):
    def __init__(self, clip_dim: int = 512, text_dim: int = 768, d_model: int = 256, nhead: int = 4, dropout: float = 0.1):
        super(MultimodalSceneEncoder, self).__init__()
        self.proj_visual = nn.Linear(clip_dim, d_model)
        self.proj_ocr = nn.Linear(text_dim, d_model)
        self.proj_transcript = nn.Linear(text_dim, d_model)
        
        self.norm_v = nn.LayerNorm(d_model)
        self.norm_o = nn.LayerNorm(d_model)
        self.norm_t = nn.LayerNorm(d_model)
        
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model)
        )
        self.norm_final = nn.LayerNorm(d_model)
        
    def forward(self, visual_emb: torch.Tensor, ocr_emb: torch.Tensor, transcript_emb: torch.Tensor):
        v = self.norm_v(self.proj_visual(visual_emb))
        o = self.norm_o(self.proj_ocr(ocr_emb))
        t_proj = self.norm_t(self.proj_transcript(transcript_emb))
        
        q = v.unsqueeze(1)
        k = torch.stack([o, t_proj], dim=1)
        v_val = k
        
        attn_output, _ = self.cross_attn(query=q, key=k, value=v_val)
        attn_output = attn_output.squeeze(1)
        
        x = self.norm_final(v + attn_output)
        joint_emb = x + self.ffn(x)
        return joint_emb, t_proj

def contrastive_loss(joint_embeddings, text_proj_embeddings, temp=0.07):
    joint_norm = F.normalize(joint_embeddings, p=2, dim=-1)
    text_norm = F.normalize(text_proj_embeddings, p=2, dim=-1)
    logits = torch.matmul(joint_norm, text_norm.T) / temp
    labels = torch.arange(logits.size(0)).to(logits.device)
    loss_v2t = F.cross_entropy(logits, labels)
    loss_t2v = F.cross_entropy(logits.T, labels)
    return (loss_v2t + loss_t2v) / 2

# Training Fusion Model
fusion_model = MultimodalSceneEncoder().to(device)
optimizer_fusion = optim.AdamW(fusion_model.parameters(), lr=1e-4, weight_decay=1e-2)

epochs_fusion = 20
losses_train_fusion, losses_val_fusion = [], []

print("[Train] Huấn luyện Thành phần 3: MultimodalSceneEncoder (Fusion Layer)...")
for epoch in range(1, epochs_fusion + 1):
    fusion_model.train()
    total_loss = 0.0
    for batch in train_loader:
        v = batch["visual"].to(device)
        o = batch["ocr"].to(device)
        t = batch["transcript"].to(device)
        
        optimizer_fusion.zero_grad()
        joint_emb, t_proj = fusion_model(v, o, t)
        loss = contrastive_loss(joint_emb, t_proj)
        loss.backward()
        optimizer_fusion.step()
        total_loss += loss.item() * v.size(0)
        
    avg_train = total_loss / len(train_dataset)
    losses_train_fusion.append(avg_train)
    
    fusion_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            v = batch["visual"].to(device)
            o = batch["ocr"].to(device)
            t = batch["transcript"].to(device)
            joint_emb, t_proj = fusion_model(v, o, t)
            val_loss += contrastive_loss(joint_emb, t_proj).item() * v.size(0)
    avg_val = val_loss / len(val_dataset)
    losses_val_fusion.append(avg_val)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs_fusion:02d}] -> Train InfoNCE: {avg_train:.4f} | Val InfoNCE: {avg_val:.4f}")

# Lưu trọng số Scene Encoder
torch.save(fusion_model.state_dict(), models_dir / "scene_encoder.pth")
print(f"[Saved] Đã lưu thành công: {models_dir / 'scene_encoder.pth'}")

## Bước 5: Trực quan hóa Lịch sử Huấn luyện & Đánh giá Đồ thị Loss

Vẽ biểu đồ so sánh hội tụ của cả 3 thành phần nhẹ vừa huấn luyện.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: KeyframeMatcher Loss
axes[0].plot(range(1, epochs_kf + 1), losses_train_kf, label="Train BCE", marker="o", color="b")
axes[0].plot(range(1, epochs_kf + 1), losses_val_kf, label="Val BCE", marker="x", color="cyan")
axes[0].set_title("Component 1: KeyframeMatcher Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()
axes[0].grid(True)

# Subplot 2: SlideTextMatcher Loss
axes[1].plot(range(1, epochs_slide + 1), losses_train_slide, label="Train BCE", marker="o", color="g")
axes[1].plot(range(1, epochs_slide + 1), losses_val_slide, label="Val BCE", marker="x", color="lime")
axes[1].set_title("Component 2: SlideTextMatcher Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("BCE Loss")
axes[1].legend()
axes[1].grid(True)

# Subplot 3: MultimodalSceneEncoder Loss
axes[2].plot(range(1, epochs_fusion + 1), losses_train_fusion, label="Train InfoNCE", marker="o", color="r")
axes[2].plot(range(1, epochs_fusion + 1), losses_val_fusion, label="Val InfoNCE", marker="x", color="orange")
axes[2].set_title("Component 3: MultimodalSceneEncoder Loss")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("InfoNCE Loss")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(outputs_dir / "modular_components_training_loss.png", dpi=150)
plt.show()

print("[Done] Tất cả các thành phần nhẹ (Visual Matchers & Fusion Network) đã được huấn luyện & kiểm tra thành công!")

## Bước 6: Kiểm tra Tích hợp Hệ thống (End-to-End Pipeline Verification) + Quality Gates cho vấn đề #5, #7, #8

Dưới đây là phiên bản pipeline suy luận có bổ sung **Quality Gate + Fallback Policy** để xử lý các điểm yếu:

- **Vấn đề #5 - ASR chất lượng thấp:**
  - Gắn cờ khi `asr_confidence` thấp hoặc `silence_ratio` cao.
  - Khi fail gate: giảm độ tin cậy transcript, ưu tiên chạy lại ASR/chunk ngắn hơn, và hạ mức trả lời ("insufficient evidence").

- **Vấn đề #7 - Mất thông tin slide quan trọng ở visual pipeline:**
  - Gắn cờ khi điểm khớp keyframe-transcript hoặc OCR-transcript thấp.
  - Khi fail gate: chuyển sang text-first mode (ưu tiên transcript/OCR), yêu cầu re-keyframe/re-OCR trước khi dùng kết quả visual cho summary.

- **Vấn đề #8 - Semantic/caption có nguy cơ hallucination:**
  - Chỉ cho phép semantic claims đi tiếp khi có grounding score đủ cao từ bằng chứng transcript + OCR + visual matching.
  - Khi fail gate: bật chế độ constrained generation (chỉ nêu thông tin có bằng chứng), và trả về cảnh báo mức rủi ro.

Mục tiêu là biến pipeline từ "always-pass" thành **"evidence-aware pipeline"** trước khi sang speaker/timeline/summarizer/Q&A.

In [ ]:
# Kiểm tra nạp lại trọng số và inference thử nghiệm + quality-gated pipeline
kf_eval = KeyframeMatcher().to(device)
kf_eval.load_state_dict(torch.load(models_dir / "keyframe_matcher.pth", map_location=device))
kf_eval.eval()

slide_eval = SlideTextMatcher().to(device)
slide_eval.load_state_dict(torch.load(models_dir / "slide_matcher.pth", map_location=device))
slide_eval.eval()

scene_eval = MultimodalSceneEncoder().to(device)
scene_eval.load_state_dict(torch.load(models_dir / "scene_encoder.pth", map_location=device))
scene_eval.eval()


def clamp01(x: float) -> float:
    return float(max(0.0, min(1.0, x)))


def quality_gate_policy(metrics: dict, thresholds: dict):
    # Gate #5: Audio/ASR quality
    asr_ok = (
        metrics["asr_confidence"] >= thresholds["min_asr_confidence"]
        and metrics["silence_ratio"] <= thresholds["max_silence_ratio"]
    )

    # Gate #7: Visual pipeline reliability
    visual_ok = (
        metrics["keyframe_transcript_score"] >= thresholds["min_keyframe_score"]
        and metrics["ocr_transcript_score"] >= thresholds["min_ocr_score"]
    )

    # Gate #8: Semantic grounding / anti-hallucination
    semantic_ok = metrics["semantic_grounding_score"] >= thresholds["min_semantic_grounding"]

    actions = []

    if not asr_ok:
        actions.append(
            "[#5] ASR gate FAIL -> re-run ASR with smaller chunks / better model; mark transcript as low-trust; limit downstream answer style to insufficient-evidence mode."
        )

    if not visual_ok:
        actions.append(
            "[#7] Visual gate FAIL -> switch to text-first mode (transcript + OCR), skip hard visual claims, queue re-keyframe/re-OCR job."
        )

    if not semantic_ok:
        actions.append(
            "[#8] Semantic gate FAIL -> enable constrained generation: only output claims linked to transcript/OCR evidence; add hallucination warning."
        )

    final_decision = {
        "asr_ok": asr_ok,
        "visual_ok": visual_ok,
        "semantic_ok": semantic_ok,
        "allow_summarization": asr_ok and semantic_ok,
        "allow_visual_claims": visual_ok and semantic_ok,
        "actions": actions if actions else ["All gates PASS -> normal multimodal summarization path."]
    }
    return final_decision


# Thao tác suy luận trên 1 sample ngẫu nhiên
sample = val_dataset[0]
v_sample = sample["visual"].unsqueeze(0).to(device)
o_sample = sample["ocr"].unsqueeze(0).to(device)
t_sample = sample["transcript"].unsqueeze(0).to(device)

with torch.no_grad():
    score_kf = clamp01(kf_eval(v_sample, t_sample).item())
    score_slide = clamp01(slide_eval(o_sample, t_sample).item())
    joint_vector, _ = scene_eval(v_sample, o_sample, t_sample)

# Proxy metrics for quality gates in this demo notebook
transcript_std = float(t_sample.std().item())
metrics = {
    "asr_confidence": clamp01(0.55 * score_slide + 0.45 * score_kf),
    "silence_ratio": clamp01(max(0.0, 0.30 - transcript_std / 5.0)),
    "keyframe_transcript_score": score_kf,
    "ocr_transcript_score": score_slide,
    "semantic_grounding_score": clamp01(0.40 * score_kf + 0.60 * score_slide),
}

thresholds = {
    "min_asr_confidence": 0.60,
    "max_silence_ratio": 0.25,
    "min_keyframe_score": 0.55,
    "min_ocr_score": 0.60,
    "min_semantic_grounding": 0.65,
}

decision = quality_gate_policy(metrics, thresholds)

print("==================================================")
print("KẾT QUẢ SUY LUẬN TÍCH HỢP + QUALITY GATES")
print("==================================================")
print(f"1. Keyframe-Transcript score: {metrics['keyframe_transcript_score']:.4f}")
print(f"2. OCR-Transcript score:      {metrics['ocr_transcript_score']:.4f}")
print(f"3. ASR confidence (proxy):    {metrics['asr_confidence']:.4f}")
print(f"4. Silence ratio (proxy):     {metrics['silence_ratio']:.4f}")
print(f"5. Semantic grounding score:  {metrics['semantic_grounding_score']:.4f}")
print(f"6. Scene joint embedding:     {joint_vector.shape} | L2 Norm: {torch.norm(joint_vector).item():.4f}")
print("--------------------------------------------------")
print(f"ASR OK?              {decision['asr_ok']}")
print(f"Visual OK?           {decision['visual_ok']}")
print(f"Semantic OK?         {decision['semantic_ok']}")
print(f"Allow Summarization? {decision['allow_summarization']}")
print(f"Allow Visual Claims? {decision['allow_visual_claims']}")
print("--------------------------------------------------")
print("Recommended actions:")
for i, action in enumerate(decision["actions"], start=1):
    print(f"  {i}. {action}")
print("==================================================")